# Chapter 7 — Diffusion and Spectral Propagation

Companion notebook.

Reproduces:
- Figure 7.1: Laplacian spectrum of a small affinity graph.
- Figure 7.2: Spectral filter shapes for heat, regularised Laplacian, PPR, p-step, Chebyshev.
- Figure 7.3: Multi-hop reasoning — diffusion kernels reach beyond one-hop neighbours.
- Figure 7.4: Downstream KRR with each diffusion kernel.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from tabkernels.diffusion import (
    HeatKernel, RegLaplacianKernel, PPRKernel, PStepKernel,
    ChebyshevKernel, CommuteTimeKernel, LearnableSpectralKernel,
    laplacian_sym, transition_sym,
)

torch.manual_seed(42); np.random.seed(42)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## Build a small affinity graph

In [ ]:
N = 30
X = torch.randn(N, 2)
sq = ((X[:, None] - X[None, :]) ** 2).sum(-1)
W = torch.exp(-sq / 0.6)
W = (W + W.T) / 2  # symmetric
L = laplacian_sym(W)
eigvals, eigvecs = torch.linalg.eigh(L)
print(f'Eigenvalue range of L_sym: [{eigvals.min().item():.3f}, {eigvals.max().item():.3f}]')

## Figure 7.1: Laplacian spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(eigvals.numpy(), 'o-')
axes[0].set_xlabel('eigenvalue index'); axes[0].set_ylabel('lambda')
axes[0].set_title('Eigenvalue spectrum of L_sym'); axes[0].grid(alpha=0.3)
for i, ax in enumerate([axes[1]]):
    sc = ax.scatter(X[:, 0], X[:, 1], c=eigvecs[:, 1], cmap='RdBu', s=40)
    ax.set_title('Second eigenvector u_2 (principal direction)')
    plt.colorbar(sc, ax=ax)
fig.suptitle('Figure 7.1: Laplacian spectrum and principal eigenvector')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_07_01_laplacian.pdf', bbox_inches='tight')
plt.show()

## Figure 7.2: Spectral filter shapes

In [ ]:
lam = np.linspace(0, 2, 200)
filters = {
    'Heat (t=0.5)':                np.exp(-0.5 * lam),
    'Heat (t=2.0)':                np.exp(-2.0 * lam),
    'Reg-Laplacian (alpha=0.5)':    1.0 / (1 + 0.5 * lam),
    'Reg-Laplacian (alpha=2.0)':    1.0 / (1 + 2.0 * lam),
    'PPR (alpha=0.1)':             0.1 / (1 - 0.9 * (1 - lam)),
    'p-step (p=2) on P_sym':       (1 - lam) ** 2,
    'Commute-time': np.where(lam > 1e-3, 1.0 / np.maximum(lam, 1e-3), 0.0),
}
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for name, f in filters.items():
    ax.plot(lam, f, label=name, lw=1.5)
ax.set_xlabel('lambda (eigenvalue of L_sym)'); ax.set_ylabel('f(lambda)')
ax.set_title('Figure 7.2: Spectral filter shapes')
ax.set_ylim(0, 5); ax.set_xlim(0, 2)
ax.legend(loc='upper right', fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_07_02_spectral_filters.pdf', bbox_inches='tight')
plt.show()

## Figure 7.3: Multi-hop reasoning

Compare the original 1-hop affinity W to a 5-hop diffusion kernel.  The diffusion kernel reaches points that are not direct neighbours in W.

In [ ]:
# Original (after sparsification): keep top-3 neighbours per row.
_, idx = W.topk(3, dim=-1)
W_sparse = torch.zeros_like(W)
rows = torch.arange(N).unsqueeze(1).expand_as(idx)
W_sparse[rows, idx] = W[rows, idx]
W_sparse = (W_sparse + W_sparse.T) / 2

# Diffuse: heat kernel at t=2.
K_heat = HeatKernel(t=2.0)(W_sparse)
# Look at row 0: 1-hop vs diffusion.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, M, title in zip(axes, [W_sparse, K_heat], ['1-hop affinity (W)', '5-hop heat kernel (exp(-2L))']):
    sc = ax.scatter(X[:, 0], X[:, 1], c=M[0].detach(), cmap='viridis', s=80, edgecolors='k')
    ax.scatter(X[0, 0], X[0, 1], c='red', s=200, marker='*', edgecolors='k', label='query')
    ax.set_title(title); ax.legend(loc='upper left')
    plt.colorbar(sc, ax=ax)
fig.suptitle('Figure 7.3: Multi-hop reasoning via diffusion (red star = query)')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_07_03_multihop.pdf', bbox_inches='tight')
plt.show()

## Figure 7.4: Downstream prediction with each diffusion kernel

Use each diffusion kernel as the prediction kernel in NW (kernel-weighted average) on a synthetic regression.

In [ ]:
# Generate y as a smooth function of the second eigenvector of L (so diffusion should help).
y = eigvecs[:, 1] + 0.1 * torch.randn(N)

kernels = {
    'Heat (t=1)':     HeatKernel(t=1.0),
    'Reg-Lap (a=1)':  RegLaplacianKernel(alpha=1.0),
    'PPR (a=0.1)':    PPRKernel(alpha=0.1),
    'p-step (p=3)':   PStepKernel(p=3),
    'Cheb (T_0=1)':   ChebyshevKernel(K=5, theta=torch.tensor([1.0, 0, 0, 0, 0])),
}

results = {}
for name, k in kernels.items():
    K = k(W_sparse)
    # NW prediction: weighted average using K.
    K_pos = K.clamp_min(0.0)  # ensure non-negative for NW
    denom = K_pos.sum(dim=-1, keepdim=True).clamp_min(1e-9)
    pred = (K_pos / denom) @ y
    mse = ((pred - y) ** 2).mean().item()
    results[name] = mse

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
names = list(results.keys()); mses = [results[n] for n in names]
ax.bar(range(len(names)), mses, color='C0')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=20)
ax.set_ylabel('NW prediction MSE')
ax.set_title('Figure 7.4: Downstream NW prediction with each diffusion kernel')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_07_04_diffusion_mse.pdf', bbox_inches='tight')
plt.show()
for n, m in results.items():
    print(f'  {n:18s}: MSE {m:.4f}')

**End of notebook.** Reproduces 4 figures.